## Ai agent for value investing decisions
Ai agent Ai agent to help with investing decisions. You provide the Ai agent with query of list of company's you are considering on buying and it will give an analysis and final recommendation based on whether company stock
price is cheap or expensive relative to its intrinsic value. It uses Ben Graham's value investing principles and basic formula for assessing each company's intrinsic/fair value.

On the backend side it uses of Hugging Face Ai smolagents framework for fast prototyping and use case experimentation.

In [ ]:
#---install needed packages and libraries
pip install smolagents[toolkit]

In [ ]:
# install agent default tools from smolagents
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, Tool, tool, VisitWebpageTool


In [ ]:
#----setup your environment variables-----
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [ ]:
# --- Imports ---
import yfinance as yf
import pandas as pd
from smolagents import CodeAgent, Tool, FinalAnswerTool, InferenceClientModel
import re
from google.colab import userdata

# --- Define your fetch function ---
def fetch_stock_data(tickers):
    """
    Fetch stock metrics and calculate intrinsic value for a list of tickers.
    """
    data = []
    def recommendation(intrinsic_value, price):
        return 'BUY' if intrinsic_value > price else 'HOLD'

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            pe_ratio = round(info.get('trailingPE', None), 2)
            roe = round(info.get('returnOnEquity', None), 2)
            price = round(info.get('currentPrice', None), 2)
            peg_ratio = round(info.get('trailingPegRatio', None), 2)
            ps_ratio = round(info.get('priceToSalesTrailing12Months', None), 2)
            analyst_rating = info.get('averageAnalystRating', None)
            if analyst_rating:
                analyst_rating = analyst_rating.split('-')[1]
            eps_trailing = info.get('trailingEps', None)
            earnings_growth = info.get('earningsGrowth', None)

            if earnings_growth and eps_trailing:
                g = round(earnings_growth * 100, 2)
                intrinsic_value = round(eps_trailing * (8.5 + 2 * g), 2)
            else:
                g, intrinsic_value = None, None

            if all([pe_ratio, roe, price, intrinsic_value, g]):
                data.append({
                    'Ticker': ticker,
                    'P/E': pe_ratio,
                    'P/EG': peg_ratio,
                    'P/Sales': ps_ratio,
                    'ROE': roe,
                    'growth%': g,
                    'Analyst_rating': analyst_rating,
                    'Price': price,
                    'FairValue': intrinsic_value,
                    'Diff': price - intrinsic_value,
                    'Recom': recommendation(intrinsic_value, price)
                })
        except Exception as e:
            print(f"⚠️ Error fetching {ticker}: {e}")
            continue

    return pd.DataFrame(data)

# --- Convert to smolagents Tool ---
class FetchStockDataTool(Tool):
    name = "fetch_stock_data"
    description = "Fetch financial metrics and intrinsic value for a list of stock tickers."
    inputs = {"tickers": {"type": "array", "description": "List of stock ticker symbols"}}
    output_type = "string"

    def forward(self, tickers):
        df = fetch_stock_data(tickers)
        return df

# --- Create a Google Search tool that extracts tickers from results ---
class GoogleTickerSearchTool(Tool):
    name = "google_ticker_search"
    description = "Use Google search to find stock ticker symbols for given company names."
    inputs = {"query": {"type": "string", "description": "User query or company names"}}
    output_type = "array"

    def forward(self, query):
        search = GoogleSearchTool()
        results = search.forward(query)
        tickers = list(set(re.findall(r'\b[A-Z]{1,5}\b', str(results))))
        return tickers

# --- Build your Agent ---

agent = CodeAgent(
    tools=[GoogleTickerSearchTool(), FetchStockDataTool(), FinalAnswerTool()],
    model=InferenceClientModel(model_id="gpt-4o-mini", provider="openai", api_key=os.environ.get("OPENAI_API_KEY")), # using OpenAI model
    instructions=
        "You are an AI financial assistant. "
        "When the user asks about stock data for companies, "
        "first use GoogleTickerSearchTool to find their ticker symbols. "
        "Then use FetchStockDataTool with the tickers to return a financial summary.",
    verbosity_level=2
)

# --- Example run ---
query = "Get intrinsic value metrics for Apple, Visa, Microsoft and Adobe"
response = agent.run(query)
pd.DataFrame(response)
